# 如何添加线程级持久化（函数式 API）

!!! info "先决条件"

    本指南假设您熟悉以下内容：
    
    - [函数式 API](../../concepts/functional_api/)
    - [持久化](../../concepts/persistence/)
    - [内存](../../concepts/memory/)
    - [聊天模型](https://python.langchain.com/docs/concepts/chat_models/)

!!! info "LangGraph API 用户无需此操作"

    如果您使用的是 LangGraph API，则无需手动实现检查点。该 API 会自动为您处理检查点。本指南在您自行实现 LangGraph 的自定义服务器时才相关。

许多 AI 应用都需要内存来在同一个 [线程](../../concepts/persistence#threads) 的多次交互中共享上下文（例如，对话中的多个回合）。在 LangGraph 函数式 API 中，可以通过 [线程级持久化](https://langchain-ai.github.io/langgraph/concepts/persistence) 将此类内存添加到任何 [entrypoint()][langgraph.func.entrypoint] 工作流中。

创建 LangGraph 工作流时，可以通过使用 [检查点](https://langchain-ai.github.io/langgraph/reference/checkpoints/#basecheckpointsaver) 来设置它以持久化其结果：


1. 创建一个检查点实例：

    ```python
    from langgraph.checkpoint.memory import InMemorySaver
    
    checkpointer = InMemorySaver()       
    ```

2. 将 `checkpointer` 实例传递给 `entrypoint()` 装饰器：

    ```python
    from langgraph.func import entrypoint
    
    @entrypoint(checkpointer=checkpointer)
    def workflow(inputs)
        ...
    ```

3. 可选地在工作流函数签名中公开 `previous` 参数：

    ```python
    @entrypoint(checkpointer=checkpointer)
    def workflow(
        inputs,
        *,
        # 你可以根据需要选择在工作流函数签名中指定 `previous`
        # 以访问上次执行时工作流的返回值
        previous
    ):
        previous = previous or []
        combined_inputs = previous + inputs
        result = do_something(combined_inputs)
        ...
    ```

4. 可选地选择哪些值将从工作流返回，哪些将由检查点保存为 `previous`：

    ```python
    @entrypoint(checkpointer=checkpointer)
    def workflow(inputs, *, previous):
        ...
        result = do_something(...)
        return entrypoint.final(value=result, save=combine(inputs, result))
    ```

本指南将展示如何为您的工作流添加线程级持久化。

!!! tip "注意"

    如果您需要 __跨多个对话或用户（跨线程持久化）__ 共享的内存，请参阅此 [操作指南](../cross-thread-persistence-functional)。

!!! tip "注意"

    如果您需要为 `StateGraph` 添加线程级持久化，请参阅此 [操作指南](../persistence)。

## 设置

首先，我们需要安装所需的软件包

In [1]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_anthropic

接下来，我们需要为 Anthropic（我们将使用的 LLM）设置 API 密钥。

In [ ]:
import getpass
import os


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("ANTHROPIC_API_KEY")

<div class="admonition tip">
    <p class="admonition-title">为 LangGraph 开发设置 <a href="https://smith.langchain.com">LangSmith</a></p>
    <p style="padding-top: 5px;">
        注册 LangSmith，以便快速发现问题并优化 LangGraph 项目的性能。LangSmith 允许您利用追踪数据来调试、测试和监控使用 LangGraph 构建的大型语言模型 (LLM) 应用——在此处 <a href="https://docs.smith.langchain.com">阅读更多关于如何开始的介绍</a>。
    </p>
</div>

## 示例：具有短期记忆的简单聊天机器人

我们将使用一个包含单个任务的工作流程，该任务会调用一个 [聊天模型](https://python.langchain.com/docs/concepts/chat_models/)。

首先，我们来定义将要使用的模型：

In [3]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-3-5-sonnet-latest")

现在我们可以定义我们的任务和工作流。为了实现持久化，我们需要将一个 [Checkpointer](https://langchain-ai.github.io/langgraph/reference/checkpoints/#langgraph.checkpoint.base.BaseCheckpointSaver) 传递给 [entrypoint()][langgraph.func.entrypoint] 装饰器。

In [4]:
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver


@task
def call_model(messages: list[BaseMessage]):
    response = model.invoke(messages)
    return response


checkpointer = InMemorySaver()


@entrypoint(checkpointer=checkpointer)
def workflow(inputs: list[BaseMessage], *, previous: list[BaseMessage]):
    if previous:
        inputs = add_messages(previous, inputs)

    response = call_model(inputs).result()
    return entrypoint.final(value=response, save=add_messages(inputs, response))

如果我们尝试使用这个工作流，对话的上下文将在交互之间持续保留：

!!! note 注意

    如果您正在使用 LangGraph Platform 或 LangGraph Studio，则 __无需__ 将 `checkpointer` 传递给入口装饰器，因为它会自动完成。

现在我们可以与代理进行交互，并看到它记住了之前的消息！

In [5]:
config = {"configurable": {"thread_id": "1"}}
input_message = {"role": "user", "content": "hi! I'm bob"}
for chunk in workflow.stream([input_message], config, stream_mode="values"):
    chunk.pretty_print()

================================== Ai Message ==================================

Hi Bob! I'm Claude. Nice to meet you! How are you today?


您随时可以恢复之前的对话：

In [6]:
input_message = {"role": "user", "content": "what's my name?"}
for chunk in workflow.stream([input_message], config, stream_mode="values"):
    chunk.pretty_print()

================================== Ai Message ==================================

Your name is Bob.


如果我们想开始一个新的对话，我们可以传递一个不同的 `thread_id`。咻！所有的记忆都没了！

In [7]:
input_message = {"role": "user", "content": "what's my name?"}
for chunk in workflow.stream(
    [input_message],
    {"configurable": {"thread_id": "2"}},
    stream_mode="values",
):
    chunk.pretty_print()

================================== Ai Message ==================================

I don't know your name unless you tell me. Each conversation I have starts fresh, so I don't have access to any previous interactions or personal information unless you share it with me.


!!! tip "流式输出 token"

    如果你想从聊天机器人中流式输出大语言模型（LLM）的 token，可以使用 `stream_mode="messages"`。请查看这篇[操作指南](../streaming-tokens)了解更多信息。